# Runner: a queue of Signpost-D notebooks, each in its own kernel

Runs the queue below, one notebook after another, with **papermill**. Each run starts its own kernel, and the GPU is freed when that kernel exits, so no restart is needed between runs. This notebook itself never uses the GPU.

- **The queue (option 8 of the plan, see the ledger):**
  1. the encoder–decoder (`09`, `MODE = "encdec"`);
  2. its control, the decoder alone (`09`, `MODE = "decoder"`).
- **Failures:** a run that fails is logged, and the next one still runs.
- **Outputs:** each run writes `<run>_output.ipynb` (the executed notebook), `<run>_results.json`, `<run>_run.log` and its adapters to the working folder. A summary table is printed after every run, and `runner_log.txt` keeps the progress.
- **Time budget:** a run is skipped if the time left is under its estimate.
- **Gated decoders:** if a Kaggle secret `HF_TOKEN` exists, it is passed to the runs through the environment, never written down. The default decoder, Qwen3-0.6B, needs no token.

Start it in a fresh session (the GPU must be free), then **Run all**.

In [ ]:
# Cell 1 — Setup: clone the branch, install papermill, read the optional Kaggle secret

import json, os, subprocess, time, traceback
from pathlib import Path

!pip install -q papermill

BRANCH = "signpost-typed-decisions"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = WORK / "llms-the-hard-way"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, "https://github.com/skamalj/llms-the-hard-way.git", str(REPO)],
                   check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
print("branch at", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

try:                                            # optional: only a gated decoder needs it; never printed or saved
    from kaggle_secrets import UserSecretsClient
    for label in ("HF_TOKEN", "hf_token"):
        try:
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret(label)
            print(f"HF_TOKEN: found in Kaggle secrets (label {label})")
            break
        except Exception:
            pass
    else:
        print("HF_TOKEN: none (fine for ungated decoders)")
except ImportError:
    print("HF_TOKEN: not on Kaggle (fine for ungated decoders)")

import papermill as pm

LOG = WORK / "runner_log.txt"


def note(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG, "a") as fh:
        fh.write(line + "\n")

In [ ]:
# Cell 2 — The queue and the time budget

BUDGET_MIN = 170                                # stop starting new runs after this many minutes in all
QUEUE = [
    {"run": "09_encdec", "notebook": "09_signpost_d_encdec.ipynb", "estimate_min": 75,
     "parameters": {"MODE": "encdec", "RUN_NAME": "09_encdec"}},
    {"run": "09_decoder", "notebook": "09_signpost_d_encdec.ipynb", "estimate_min": 45,
     "parameters": {"MODE": "decoder", "RUN_NAME": "09_decoder"}},
]
for q in QUEUE:
    print(q["run"], q["parameters"])

In [ ]:
# Cell 3 — Run the queue; a summary after every run

T0 = time.time()
STATUS = {}


def summary():
    rows = []
    for q in QUEUE:
        r, f = q["run"], WORK / f"{q['run']}_results.json"
        if not f.exists():
            rows.append(f"{r:12s} {STATUS.get(r, 'not run')}")
            continue
        res = json.load(open(f))
        dev = res["dev_after"]["T per type"]["all"]
        parts = res.get("dev_parts", {})
        v1 = parts.get("v1 cases (as run 6)", {})
        inv = parts.get("added cases (invoices)", {})
        b = res["benchmark"]["T per type"]
        rows.append(f"{r:12s} {res['parameters_m']:>5} M | dev acc {dev['accuracy']:.3f} KL {dev['kl']:.3f} | "
                    f"dev v1 {v1.get('accuracy', 0):.3f}/{v1.get('kl', 0):.3f} | dev invoices {inv.get('accuracy', 0):.3f}/{inv.get('kl', 0):.3f} | "
                    f"bench acc {b['all']['accuracy']:.4f} KL {b['all']['kl_from_gold']:.4f} | "
                    f"invoices {b.get('workflow=invoice_processing', {}).get('accuracy', 0):.3f}")
        for k, v in res.get("field_flip", {}).items():
            rows.append(f"{'':12s}   field-flip {k:42s} mean change {v['mean_tv']:.3f} | top changed {v['top_changed']}/{v['cases']}")
    text = "\n".join(["run           params | our dev (T per type) | v1 part acc/KL | invoice part acc/KL | benchmark (T per type)",
                       *rows, "references: run 7 (08) dev v1 0.713/0.430, bench 0.471 / KL 0.548, invoices 0.478; prior 0.470 / 0.347; run 7 field-flip: agent outcome 0.758 60/60, agent action 0.163 0/60, invoice 0.005 0/60, security 0.022 1/60"])
    (WORK / "runner_summary.txt").write_text(text)
    print(text)


for q in QUEUE:
    used = (time.time() - T0) / 60
    if used + q["estimate_min"] > BUDGET_MIN:
        STATUS[q["run"]] = f"skipped: {used:.0f} min used, needs about {q['estimate_min']}"
        note(f"{q['run']}: {STATUS[q['run']]}")
        continue
    note(f"{q['run']}: starting ({used:.0f} min used so far)")
    t = time.time()
    try:
        pm.execute_notebook(str(REPO / q["notebook"]), str(WORK / f"{q['run']}_output.ipynb"), parameters=q["parameters"],
                            cwd=str(WORK), kernel_name="python3", log_output=False, progress_bar=False)
        STATUS[q["run"]] = f"done in {(time.time() - t) / 60:.0f} min"
    except Exception as e:                        # one failed run must not stop the queue
        STATUS[q["run"]] = f"FAILED after {(time.time() - t) / 60:.0f} min: {type(e).__name__}"
        note(f"{q['run']}: failed\n{traceback.format_exc()[-3000:]}")
    note(f"{q['run']}: {STATUS[q['run']]}")
    summary()
note(f"queue finished in {(time.time() - T0) / 60:.0f} min")